# 00 · Setup and preflight

**Backup plan, step 1 of 4.** The comparison runs on *this* machine: the Left side is read with the
usual `dtrack` sources, the Right side is read from a Databricks SQL warehouse, and both are compared
with the same statistics code. Stage 1 counts and Stage 2 deterministic sample predicates execute
inside each source/warehouse backend, so only sampled rows are normally pulled here.
Set `PUSH_SAMPLE=False` only to retain the full Right window locally. Left key families sample
source-side; a custom transform without backend SQL support fails before any full-window pull.

| Notebook | Does | Writes |
| --- | --- | --- |
| 00 | environment, warehouse, preflight | `OUT/LATEST`, `<run>/` |
| 01 | Stage 1 row counts, reconcile, gate | `row_check/`, `manifest/` |
| 02 | Stage 2 pulls, statistics, comparison | `_INDEX/`, `units/`, `results/`, `_local/data.duckdb` |
| 03 | report, publish, S3 backup | `report/`, report store, `s3://.../runs/<run_id>/` |
| 04 | roll one pair back after a config fix | deletes that pair's signals |

The run folder has the same layout as a distributed run on S3, so every step can be re-run: work
already done is skipped. `python -m dtrack_local run --config c.json` does all four in one go.

**Before you start**

- Install: `uv sync --extra dbx` (plus `--extra ldap` / `athena` if the Left side needs them).
- `.env` (found from the working directory upwards):
  - `DTRACK_DBX_PROFILE` (default `corp`, a `~/.databrickscfg` profile) and `DTRACK_DBX_WAREHOUSE_ID`
  - Left credentials, e.g. `<conn_macro>_USR / _PWD / _DSN`
  - optional `DTRACK_REPORT_S3` (report store) and `DTRACK_LOCAL_BACKUP_S3` (runs root for the backup)
- Right-side names in `col_map` must match the Databricks spelling (see `DESCRIBE`).

In [ ]:
# Parameters: keep CONFIG / OUT identical in notebooks 00-03
CONFIG = "c.json"  # pairs config (col_map files resolve next to it)
OUT = ".dtrack-local/runs"  # local runs root; one folder per run id
RUN_ID = None  # None: 00 starts a new run, 01-03 continue the run 00 opened (OUT/LATEST)
DISPOSITION = (
    "external_links"  # "inline" when presigned downloads are blocked (<= 25 MiB per window)
)
PUSH_SAMPLE = True  # False only when you need to land the full Right window locally
PAIRS = None  # e.g. ["orders"]; None = every active pair

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 40, "display.width", 200)

from dtrack_local import pipeline

## Environment
Names only; values are never printed.

In [ ]:
names = [
    "DTRACK_DBX_PROFILE",
    "DTRACK_DBX_WAREHOUSE_ID",
    "DTRACK_REPORT_S3",
    "DTRACK_LOCAL_BACKUP_S3",
]
pd.DataFrame({"variable": names, "set": [bool(os.getenv(name)) for name in names]})

## Databricks: who am I, and is the warehouse up
A stopped warehouse also starts on the first statement; starting it here makes the wait visible.

In [ ]:
from dtrack_left.backends import databricks

w = databricks.workspace(os.getenv("DTRACK_DBX_PROFILE") or databricks.DEFAULT_PROFILE)
print(json.dumps(databricks.login(w, open_browser=False), indent=2))
WAREHOUSE_ID = os.environ["DTRACK_DBX_WAREHOUSE_ID"]
status = databricks.warehouse_status(w, WAREHOUSE_ID)
if status["state"] != "RUNNING":
    status = databricks.warehouse_start(w, WAREHOUSE_ID, wait=True, timeout_minutes=20)
pd.Series(status)

## Open the run
`RUN_ID = None` starts a new run and records it in `OUT/LATEST` for notebooks 01-03.

In [ ]:
run = pipeline.open_run(
    CONFIG, run_id=RUN_ID, out=OUT, disposition=DISPOSITION, push_sample=PUSH_SAMPLE, pairs=PAIRS
)
Path(OUT).mkdir(parents=True, exist_ok=True)
(Path(OUT) / "LATEST").write_text(run.run_id)
print("run_id:", run.run_id)
print("folder:", run.folder.resolve())
print("error log:", run.error_log.resolve())

## What will be compared

In [ ]:
from dtrack_left.cli.tools import describe_right

rows = []
for name, pair in run.config["pairs"].items():
    if pair.get("skip") or (PAIRS and name not in PAIRS):
        continue
    left = pair["left"]
    rows.append(
        {
            "pair": name,
            "left": f"{left.get('source')}: {left.get('table') or left.get('path') or 'query'}",
            "right": describe_right(pair["right"])["target"],
            "dates": f"{pair.get('fromDate') or '-'} .. {pair.get('toDate') or '-'}",
            "vintage": pair.get("vintage") or "day",
            "sample": pair.get("sample") or "all",
            "key": pair.get("key") or "",
            "columns": len(pair["col_map"]),
        }
    )
pd.DataFrame(rows)

## Preflight
Left: `SELECT * ... WHERE 1=0` on the source. Right: `DESCRIBE QUERY` through the warehouse. Both must be OK before notebook 01.

In [ ]:
health = pipeline.preflight(run)
table = pd.DataFrame(
    [
        {"pair": name, "side": side, "ok": row[side].get("ok"), "error": row[side].get("error", "")}
        for name, row in health.items()
        for side in ("left", "right")
    ]
)
display(table)
assert table["ok"].all(), "fix the failing side before notebook 01"